In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"

reports_dir.mkdir(parents=True, exist_ok=True)

df_menu = pd.read_csv(
    processed_dir / "menus_clean.csv"
)

df_branches = pd.read_csv(
    processed_dir / "branches_clean.csv"
)

df_info = pd.read_csv(
    processed_dir / "general_information_clean.csv"
)

df_actual = pd.read_csv(
    processed_dir / "survey_actual.csv"
)

df_dummy = pd.read_csv(
    processed_dir / "survey_dummy.csv"
)

df_sales = pd.read_csv(
    processed_dir / "coffee_shop_sales_clean.csv"
)

print("=== DATASET BERHASIL DIBACA ===")

print("Menu:", len(df_menu))
print("Cabang:", len(df_branches))
print("General Information:", len(df_info))
print("Survey Actual:", len(df_actual))
print("Survey Dummy:", len(df_dummy))
print("Sales Public:", len(df_sales))

=== DATASET BERHASIL DIBACA ===
Menu: 183
Cabang: 3
General Information: 9
Survey Actual: 2
Survey Dummy: 98
Sales Public: 149116


In [3]:
validation_results = []

def add_validation(dataset, check_name, passed, details):
    validation_results.append({
        "dataset": dataset,
        "check_name": check_name,
        "status": "PASS" if passed else "FAIL",
        "details": str(details)
    })

def add_review(dataset, check_name, details):
    validation_results.append({
        "dataset": dataset,
        "check_name": check_name,
        "status": "REVIEW",
        "details": str(details)
    })

In [4]:
print("=== VALIDASI DATASET MENU ===")

add_validation(
    "menu",
    "Jumlah record",
    len(df_menu) == 183,
    f"Jumlah record: {len(df_menu)}"
)

add_validation(
    "menu",
    "Keunikan menu_id",
    df_menu["menu_id"].is_unique,
    "Pemeriksaan ID menu"
)

add_validation(
    "menu",
    "Harga positif",
    (df_menu["price"] > 0).all(),
    "Seluruh harga harus positif"
)

add_validation(
    "menu",
    "Kategori umum lengkap",
    df_menu["category_group"].notna().all(),
    "Tidak boleh ada kategori umum kosong"
)

print("Validasi menu selesai.")

=== VALIDASI DATASET MENU ===
Validasi menu selesai.


In [5]:
print("=== VALIDASI RELASI MENU-CABANG ===")

valid_branch_ids = set(
    df_branches["branch_id"].dropna()
)

invalid_menu_branches = df_menu[
    ~df_menu["branch_id"].isin(valid_branch_ids)
]

add_validation(
    "menu",
    "Relasi branch_id",
    len(invalid_menu_branches) == 0,
    f"Menu dengan branch_id tidak valid: {len(invalid_menu_branches)}"
)

display(
    invalid_menu_branches[
        ["menu_id", "branch_id", "menu_name"]
    ]
)

=== VALIDASI RELASI MENU-CABANG ===


,menu_id,branch_id,menu_name


In [6]:
print("=== VALIDASI DATASET CABANG ===")

add_validation(
    "branches",
    "Jumlah cabang",
    len(df_branches) == 3,
    f"Jumlah cabang: {len(df_branches)}"
)

add_validation(
    "branches",
    "Keunikan branch_id",
    df_branches["branch_id"].is_unique,
    "ID cabang harus unik"
)

for col in ["opening_time", "closing_time"]:
    valid_time = (
        df_branches[col]
        .astype("string")
        .str.fullmatch(r"(?:[01]\d|2[0-3]):[0-5]\d")
        .fillna(False)
    )

    add_validation(
        "branches",
        f"Format {col}",
        valid_time.all(),
        f"Jumlah format tidak valid: {(~valid_time).sum()}"
    )

add_review(
    "branches",
    "Verifikasi PIC",
    "Seluruh cabang masih memerlukan konfirmasi informasi operasional"
)

print("Validasi cabang selesai.")

=== VALIDASI DATASET CABANG ===
Validasi cabang selesai.


In [7]:
print("=== VALIDASI GENERAL INFORMATION ===")

add_validation(
    "general_information",
    "Jumlah record",
    len(df_info) == 9,
    f"Jumlah record: {len(df_info)}"
)

add_validation(
    "general_information",
    "Keunikan info_id",
    df_info["info_id"].is_unique,
    "ID informasi harus unik"
)

add_validation(
    "general_information",
    "Keunikan info_key",
    df_info["info_key"].is_unique,
    "Topik informasi harus unik"
)

valid_content = (
    df_info["content"]
    .astype("string")
    .str.strip()
    .str.len()
    .gt(0)
    .fillna(False)
)

add_validation(
    "general_information",
    "Kelengkapan konten",
    valid_content.all(),
    f"Konten kosong: {(~valid_content).sum()}"
)

add_review(
    "general_information",
    "Verifikasi sumber",
    "Kesesuaian isi informasi dengan bukti wawancara perlu diperiksa"
)

print("Validasi general information selesai.")

=== VALIDASI GENERAL INFORMATION ===
Validasi general information selesai.


In [8]:
print("=== VALIDASI CUSTOMER SURVEY ===")

add_validation(
    "survey",
    "Jumlah respons",
    len(df_actual) + len(df_dummy) == 100,
    f"Total: {len(df_actual) + len(df_dummy)}"
)

add_validation(
    "survey",
    "Label actual",
    df_actual["data_type"].eq("actual").all(),
    "Dataset actual hanya berisi respons nyata"
)

add_validation(
    "survey",
    "Label dummy",
    df_dummy["data_type"].eq("dummy").all(),
    "Dataset dummy hanya berisi simulasi"
)

overlap = set(df_actual["response_id"]) & set(
    df_dummy["response_id"]
)

add_validation(
    "survey",
    "Tidak ada ID tumpang tindih",
    len(overlap) == 0,
    f"Jumlah ID tumpang tindih: {len(overlap)}"
)

add_review(
    "survey",
    "Keterbatasan respons aktual",
    "Hanya tersedia dua respons aktual; tidak representatif"
)

print("Validasi survei selesai.")

=== VALIDASI CUSTOMER SURVEY ===
Validasi survei selesai.


In [9]:
print("=== VALIDASI SALES PUBLIC ===")

add_validation(
    "sales_public",
    "Jumlah record",
    len(df_sales) == 149116,
    f"Jumlah record: {len(df_sales)}"
)

add_validation(
    "sales_public",
    "Harga positif",
    (df_sales["unit_price"] > 0).all(),
    "Harga harus positif"
)

add_validation(
    "sales_public",
    "Kuantitas positif",
    (df_sales["transaction_qty"] > 0).all(),
    "Kuantitas harus positif"
)

expected_total = (
    df_sales["transaction_qty"] *
    df_sales["unit_price"]
)

amount_valid = (
    (df_sales["total_amount"] - expected_total)
    .abs() < 0.011
).all()

add_validation(
    "sales_public",
    "Konsistensi total_amount",
    amount_valid,
    "Nilai transaksi sesuai kuantitas dikali harga"
)

add_validation(
    "sales_public",
    "Label public",
    df_sales["data_type"].eq("public").all(),
    "Dataset tidak boleh berlabel actual Kopilogi"
)

add_review(
    "sales_public",
    "Lisensi dataset",
    "Lisensi dan sumber asli Kaggle belum dikonfirmasi"
)

print("Validasi dataset publik selesai.")

=== VALIDASI SALES PUBLIC ===
Validasi dataset publik selesai.


In [10]:
print("=== FINAL VALIDATION REPORT ===")

validation_report = pd.DataFrame(
    validation_results
)

display(validation_report)

print("\nRingkasan status:")
display(
    validation_report["status"].value_counts()
)

=== FINAL VALIDATION REPORT ===


,dataset,check_name,status,details
0,menu,Jumlah record,PASS,Jumlah record: 183
1,menu,Keunikan menu_id,PASS,Pemeriksaan ID menu
2,menu,Harga positif,PASS,Seluruh harga harus positif
3,menu,Kategori umum lengkap,PASS,Tidak boleh ada kategori umum kosong
4,menu,Relasi branch_id,PASS,Menu dengan branch_id tidak valid: 0
5,branches,Jumlah cabang,PASS,Jumlah cabang: 3
6,branches,Keunikan branch_id,PASS,ID cabang harus unik
7,branches,Format opening_time,PASS,Jumlah format tidak valid: 0
8,branches,Format closing_time,PASS,Jumlah format tidak valid: 0
9,branches,Verifikasi PIC,REVIEW,Seluruh cabang masih memerlukan konfirmasi inf...



Ringkasan status:


status
PASS      22
REVIEW     4
Name: count, dtype: int64

In [11]:
report_path = (
    reports_dir / "final_validation_report.csv"
)

issues_path = (
    reports_dir / "final_validation_issues.csv"
)

validation_report.to_csv(
    report_path,
    index=False,
    encoding="utf-8-sig"
)

validation_issues = validation_report[
    validation_report["status"].isin(
        ["FAIL", "REVIEW"]
    )
].copy()

validation_issues.to_csv(
    issues_path,
    index=False,
    encoding="utf-8-sig"
)

print("Laporan validasi berhasil disimpan:")
print(report_path)
print(issues_path)

Laporan validasi berhasil disimpan:
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\reports\final_validation_report.csv
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\reports\final_validation_issues.csv


In [12]:
print("=== PEMERIKSAAN KETERBATASAN MENU ===")

for col in ["is_best_seller", "is_available"]:
    missing_count = df_menu[col].isna().sum()

    if missing_count > 0:
        add_review(
            "menu",
            f"Kelengkapan {col}",
            f"{missing_count} record belum memiliki informasi"
        )

# Perbandingan katalog BR001 dan BR003
columns = ["menu_name", "category", "price"]

br001 = (
    df_menu[df_menu["branch_id"] == "BR001"][columns]
    .sort_values(columns)
    .reset_index(drop=True)
)

br003 = (
    df_menu[df_menu["branch_id"] == "BR003"][columns]
    .sort_values(columns)
    .reset_index(drop=True)
)

if br001.equals(br003):
    add_review(
        "menu",
        "Kesamaan katalog BR001 dan BR003",
        "Kedua katalog identik dan memerlukan verifikasi sumber"
    )

print("Pemeriksaan keterbatasan menu selesai.")

=== PEMERIKSAAN KETERBATASAN MENU ===
Pemeriksaan keterbatasan menu selesai.
